In [5]:
!pip freeze | grep triton
import triton
import triton.language as tl

triton==3.6.0


In [ ]:
@triton.jit 
def get_per_token_logps_kernel(
    logits_ptr,
    input_ids_ptr,
    per_token_logps_ptr,
    vocab_size,
    lse_max_ptr,           # saved m
    lse_sum_ptr,           # saved d
    BLOCK_SIZE: tl.constexpr
): 
    row_id = tl.program_id(0) 
    row_start = row_id * vocab_size

    target_token_id = tl.load(input_ids_ptr + row_id)
    target_token_logit = tl.load(logits_ptr + row_start + target_token_id)

    running_max = float('-inf')
    running_sum = 0
    for i in range(0, vocab_size, BLOCK_SIZE):
        cols = i + tl.arange(0, BLOCK_SIZE)
        mask = cols < vocab_size
        logits_block = tl.load(logits_ptr + row_start + cols, mask=mask, other=float('-inf'))
        m_local = tl.max(logits_block, axis=0)
        d_local = tl.sum(tl.exp(logits_block - m_local), axis=0)
        new_max = tl.maximum(running_max, m_local)
        running_sum = running_sum * tl.exp(running_max - new_max) + d_local * tl.exp(m_local - new_max)
        running_max = new_max
    log_prob = (target_token_logit - running_max) - tl.log(running_sum)
    tl.store(per_token_logps_ptr + row_id, log_prob)
    tl.store(lse_max_ptr + row_id, running_max)
    tl.store(lse_sum_ptr + row_id, running_sum)
    


    

In [ ]:
@triton.jit
def get_per_token_logps_backward_kernel(
    d_per_token_logps_ptr, # dy
    logits_ptr,            # x
    input_ids_ptr,         # target token indices
    lse_max_ptr,           # saved m
    lse_sum_ptr,           # saved d
    d_logits_ptr,          # Output: dx (gradients)
    vocab_size,
    BLOCK_SIZE: tl.constexpr
):
    row_id = tl.program_id(0)
    row_start = row_id * vocab_size
    d_per_token_logps = tl.load(d_per_token_logps_ptr + row_id)
    target_token_id = tl.load(input_ids_ptr + row_id)
    lse_max = tl.load(lse_max_ptr + row_id)
    lse_sum = tl.load(lse_sum_ptr + row_id)
    for i in range(0, vocab_size, BLOCK_SIZE):
        cols = i + tl.arange(0, BLOCK_SIZE)
        mask = cols < vocab_size
        logits_block = tl.load(logits_ptr + row_start + cols, mask=mask, other=float('-inf'))
        p_block = tl.exp(logits_block - lse_max) / lse_sum
        is_target = tl.where(cols == target_token_id, 1.0, 0.0)
        dx_block = d_per_token_logps * (is_target - p_block)
        tl.store(d_logits_ptr + row_start + cols, dx_block, mask=mask)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/giriksetya/Desktop/Girik_Coding/learning-sessions/GRPO/kernels'